#### ***What is Re-ranking***
##### **Re-ranking is a technique that improves the ordering of retrieved chunks before send to the LLM**

#### **Why Re-ranking**
##### **Re-ranking decides the which retrived chunks are deserved to passed to the LLM.**

#### ***What is CrossEncoder***

##### ***The query and documents are processed Together.***

In [77]:
### Load the environment variables

from dotenv import load_dotenv

load_dotenv()

True

In [78]:
###path Exists
import os
path = "../kubernetes"

if os.path.exists(path):
    print("Path Is Exist.")
else:
    print("Path is not Exist.")

Path Is Exist.


In [79]:
### Load all pdf files using DirectoryLoader by PyMuPdfLoader

from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader

loader = DirectoryLoader(
    path,
    glob = "*.pdf",
    loader_cls=PyMuPDFLoader
)

documents = loader.load()


print("Number Of Documents:",len(documents))

Number Of Documents: 3983


In [80]:
### Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100
)

chunks = splitter.split_documents(documents)

print("Number Of Chunks:",len(chunks))

Number Of Chunks: 12694


In [81]:
### BM25 Retriever

from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(
    documents=chunks
)

In [82]:
bm25_retriever.k=10

In [83]:
###create a embedding by using langchain hugging face.

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model_name = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [84]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name="kubernetes_rag",
    embedding_function=embedding_model
)

In [85]:
similarity_retriever = vectorstore.as_retriever(search_type = "similarity",
        search_kwargs = {"k":10}
)

In [86]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[similarity_retriever,bm25_retriever],
    weights=[0.8,0.2]
)

In [87]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [88]:
query = "What is a Kubernetes Deployment?"

retrieved_chunks = hybrid_retriever.invoke(query,k=20)

pairs = [[query,chunk.page_content]
 for chunk in retrieved_chunks] ##Combining both query and retrieved_chunks

print(len(pairs))

20


In [89]:
scores = reranker.predict(pairs)
print(len(scores))

20


In [90]:
for chunk,score in zip(retrieved_chunks,scores):
    print("Score:",score)
    print("Page:",chunk.metadata.get("page"))
    print("Page Content:",chunk.page_content[:300])
    print("-"*60)

Score: 7.9213734
Page: 5
Page Content: system and its components. The Kubernetes control plane continually and actively manages
every object's actual state to match the desired state you supplied.
For example: in Kubernetes, a Deployment is an object that can represent an application
running on your cluster. When you create the Deploymen
------------------------------------------------------------
Score: 3.9102814
Page: 0
Page Content: The Concepts section helps you learn about the parts of the Kubernetes system and the
abstractions Kubernetes uses to represent your cluster, and helps you obtain a deeper
understanding of how Kubernetes works.
Overview
Kubernetes is a portable, extensible, open source platform for managing containe
------------------------------------------------------------
Score: 3.6074126
Page: 1
Page Content: Pods with higher Priority can schedule on Nodes. Eviction is the process of proactively
terminating one or more Pods on resource-starved Nodes.
Cluster Adminis

In [91]:
ranked_chunks = sorted(zip(retrieved_chunks,scores),key = lambda x:x[1], reverse=True)

for chunk in ranked_chunks[:5]:
    print(chunk)

(Document(id='8d5c7d19-a6b3-4b4f-b7e8-75ddb6c33e27', metadata={'modDate': '', 'file_path': '..\\kubernetes\\Concepts.pdf', 'keywords': '', 'author': '', 'creationDate': '', 'format': 'PDF 1.7', 'source': '..\\kubernetes\\Concepts.pdf', 'page': 5, 'subject': '', 'total_pages': 676, 'creator': '', 'moddate': '', 'title': '', 'producer': 'WeasyPrint 56.1', 'creationdate': '', 'trapped': ''}, page_content="system and its components. The Kubernetes control plane continually and actively manages\nevery object's actual state to match the desired state you supplied.\nFor example: in Kubernetes, a Deployment is an object that can represent an application\nrunning on your cluster. When you create the Deployment, you might set the Deployment spec\nto specify that you want three replicas of the application to be running. The Kubernetes system\nreads the Deployment spec and starts three instances of your desired application--updating the\nstatus to match your spec. If any of those instances should 

In [92]:
ranked_chunks

[(Document(id='8d5c7d19-a6b3-4b4f-b7e8-75ddb6c33e27', metadata={'modDate': '', 'file_path': '..\\kubernetes\\Concepts.pdf', 'keywords': '', 'author': '', 'creationDate': '', 'format': 'PDF 1.7', 'source': '..\\kubernetes\\Concepts.pdf', 'page': 5, 'subject': '', 'total_pages': 676, 'creator': '', 'moddate': '', 'title': '', 'producer': 'WeasyPrint 56.1', 'creationdate': '', 'trapped': ''}, page_content="system and its components. The Kubernetes control plane continually and actively manages\nevery object's actual state to match the desired state you supplied.\nFor example: in Kubernetes, a Deployment is an object that can represent an application\nrunning on your cluster. When you create the Deployment, you might set the Deployment spec\nto specify that you want three replicas of the application to be running. The Kubernetes system\nreads the Deployment spec and starts three instances of your desired application--updating the\nstatus to match your spec. If any of those instances should

In [93]:
top_chunks = [chunk for chunk,score in ranked_chunks[:5]]

In [95]:
query

'What is a Kubernetes Deployment?'

In [94]:
top_chunks

[Document(id='8d5c7d19-a6b3-4b4f-b7e8-75ddb6c33e27', metadata={'modDate': '', 'file_path': '..\\kubernetes\\Concepts.pdf', 'keywords': '', 'author': '', 'creationDate': '', 'format': 'PDF 1.7', 'source': '..\\kubernetes\\Concepts.pdf', 'page': 5, 'subject': '', 'total_pages': 676, 'creator': '', 'moddate': '', 'title': '', 'producer': 'WeasyPrint 56.1', 'creationdate': '', 'trapped': ''}, page_content="system and its components. The Kubernetes control plane continually and actively manages\nevery object's actual state to match the desired state you supplied.\nFor example: in Kubernetes, a Deployment is an object that can represent an application\nrunning on your cluster. When you create the Deployment, you might set the Deployment spec\nto specify that you want three replicas of the application to be running. The Kubernetes system\nreads the Deployment spec and starts three instances of your desired application--updating the\nstatus to match your spec. If any of those instances should 

In [96]:
test_queries = [
    "What is a Kubernetes Deployment?",
    "What is a Kubernetes Pod?",
    "What is a Kubernetes Service?",
    "What is a ReplicaSet?",
    "What is a ConfigMap?",
    "Deployment spec replicas desired state status",
    "Pod containers shared network storage node",
    "Service clusterIP selector endpoints Pods",
    "How is a Deployment related to a ReplicaSet?",
    "How does a ReplicaSet maintain Pods?"
]

In [98]:
for query in test_queries:

    retrieved_docs = hybrid_retriever.invoke(query)
    pairs = [[query,doc.page_content] for doc in retrieved_docs]
    print("Query:",query)
    scores = reranker.predict(pairs)

    ranked_docs = sorted(zip(retrieved_docs,scores),key = lambda x:x[1],reverse=True)

    top_5_chunks = ranked_docs[:5]

    for rank,(chunk,score) in enumerate(top_5_chunks,start=1):
        print(
            f"Rank {rank} | Page {chunk.metadata.get('page')} | Score {score}"
        )
    print("#"*50)

Query: What is a Kubernetes Deployment?
Rank 1 | Page 5 | Score 7.92137336730957
Rank 2 | Page 30 | Score 5.550075531005859
Rank 3 | Page 2 | Score 4.845337390899658
Rank 4 | Page 32 | Score 4.564247131347656
Rank 5 | Page 3 | Score 4.559100151062012
##################################################
Query: What is a Kubernetes Pod?
Rank 1 | Page 83 | Score 8.690617561340332
Rank 2 | Page 85 | Score 6.734971046447754
Rank 3 | Page 85 | Score 6.344106674194336
Rank 4 | Page 1 | Score 5.566908836364746
Rank 5 | Page 85 | Score 5.357826232910156
##################################################
Query: What is a Kubernetes Service?
Rank 1 | Page 246 | Score 7.262441635131836
Rank 2 | Page 449 | Score 6.653943061828613
Rank 3 | Page 0 | Score 6.529098987579346
Rank 4 | Page 3 | Score 6.149477958679199
Rank 5 | Page 36 | Score 5.7401580810546875
##################################################
Query: What is a ReplicaSet?
Rank 1 | Page 27 | Score 5.242628574371338
Rank 2 | Page 24 | Score

In [ ]:
### Using the LLM to generate the response for every query using reranking and Hybrid
for query in test_queries:

    retrieved_docs = hybrid_retriever.invoke(query)
    pairs = [[query,doc.page_content] for doc in retrieved_docs]
    print("Query:",query)
    scores = reranker.predict(pairs)

    ranked_docs = sorted(zip(retrieved_docs,scores),key = lambda x:x[1],reverse=True)

    top_5_chunks = ranked_docs[:5]

    hybrid_docs = hybrid_retriever.invoke(query)[:5]

    rerank_context = "\n\n".join(doc.page_content for doc in top_5_chunks)
    hybrid_context = "\n\n".join(doc.page_content for doc in hybrid_docs)

    